# CMSC 173 &middot; Machine Learning &mdash; Week 2 Lab
## Parameter Estimation: Method of Moments, Maximum Likelihood & the Bias&ndash;Variance Tradeoff

Last week's lecture asked a plain question: given some data, how do you recover the settings
("parameters") of the distribution that produced it? You saw two answers &mdash; **Method of
Moments (MoM)** and **Maximum Likelihood (MLE)** &mdash; and the idea of the **bias&ndash;variance
tradeoff**. This lab walks you through all three, slowly, with pictures.

**How this lab works.** Each part has three pieces:
1. a short **plain-English explainer** of the idea,
2. a **code cell** you run, followed by a **line-by-line walkthrough** of what the code did,
3. an **Answer here** box for a sentence or two.

You will not derive anything by hand &mdash; the code does the maths, and we *graph* the results so
you can see what is going on. **NumPy + Matplotlib only** (no scikit-learn until week 4).

**Not graded.** About 60 minutes.

---
## Part 0 &middot; Setup

Run this once. It imports the two tools we need and fixes the random seed so your numbers
match mine.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(173)   # fixed seed -> reproducible 'random' numbers
print("Python", sys.version.split()[0], "| NumPy", np.__version__)
print("Ready.")

---
## Part 1 &middot; A moment is just an average of a power

Don't let the word "moment" scare you. It only means: take a power of each data point, then
average.

- average the data itself &rarr; the **mean**
- average the *squared* data &rarr; the **second moment**

**Method of Moments** is built on one move: the averages you compute *from your data* should
equal the averages the distribution *predicts*. That's it &mdash; no calculus.

In [ ]:
data = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])

mean = data.mean()                 # (1) average of the data
var  = ((data - mean)**2).mean()   # (2) average squared distance from the mean = variance
m2   = (data**2).mean()            # (3) average of the SQUARED data (the 'second moment')

print("mean         =", mean)
print("variance     =", round(var, 4))
print("m2 - mean**2 =", round(m2 - mean**2, 4))   # (4) notice: same as the variance

**Reading the code, line by line:**
- **(1)** `data.mean()` adds the 8 numbers and divides by 8. That's the first moment.
- **(2)** `data - mean` subtracts the mean from *every* point at once (NumPy broadcasting).
  Square them, average them &rarr; the variance (how spread out the data is).
- **(3)** `data**2` squares every point; `.mean()` averages them &rarr; the second moment.
- **(4)** `m2 - mean**2` gives the *same* number as the variance. That's a handy identity:
  variance = (average of squares) &minus; (square of the average).

**Answer here** (double-click to edit):

1. `variance` and `m2 - mean**2` printed the same value. Say **when** the `m2 - mean**2` route
   might be more convenient (hint: think about data arriving one point at a time).
   &rarr; *your answer*

2. Add a line that prints `data.var(ddof=1)` (divides by $n-1$ instead of $n$). Is it bigger or
   smaller than `var`, and does the gap look big or tiny for 8 points?
   &rarr; *your answer*

---
## Part 2 &middot; Method of Moments, on a Normal (with a picture)

Now the real thing. We *generate* 200 points from a Normal whose true mean and spread we
choose, then estimate them back with MoM &mdash; and plot the fitted curve on top of the data so
you can literally see the estimate. For a Normal, MoM is blunt: **estimated mean = sample
mean, estimated variance = sample variance**.

In [ ]:
true_mu, true_sigma = 5.0, 2.0
sample = rng.normal(true_mu, true_sigma, size=200)   # (1) 200 draws from the true Normal

def mom_normal(x):                                   # (2) our estimator, two averages
    mu_hat  = x.mean()
    var_hat = ((x - mu_hat)**2).mean()
    return mu_hat, var_hat

mu_hat, var_hat = mom_normal(sample)                 # (3) run it on our sample
print(f"true:     mu = {true_mu}, variance = {true_sigma**2:.2f}")
print(f"estimate: mu = {mu_hat:.3f}, variance = {var_hat:.3f}")

# (4) draw the data as a histogram, and the ESTIMATED Normal curve on top
xs = np.linspace(sample.min(), sample.max(), 200)
curve = np.exp(-(xs - mu_hat)**2 / (2*var_hat)) / np.sqrt(2*np.pi*var_hat)  # bell curve
plt.figure(figsize=(7, 4))
plt.hist(sample, bins=20, density=True, alpha=0.5, label='data')
plt.plot(xs, curve, 'r-', lw=2, label='estimated Normal')
plt.axvline(mu_hat, color='r', ls='--', alpha=0.7, label=f'estimated mean = {mu_hat:.2f}')
plt.title('MoM fits a Normal by matching the mean and the spread')
plt.legend(); plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** `rng.normal(mu, sigma, size=200)` draws 200 random numbers from a Normal. Because we
  picked `true_mu` and `true_sigma`, we know the right answer and can grade the estimate.
- **(2)** `mom_normal` is the estimator: `x.mean()` for the centre, and the average squared
  distance from that centre for the spread. Two averages, nothing more.
- **(3)** we call it on our sample and print the estimate next to the truth &mdash; they're close.
- **(4)** the plot: `plt.hist(..., density=True)` shows the data as bars scaled to look like a
  probability; `curve` is the bell-curve formula evaluated at the *estimated* mean and variance;
  the dashed line marks the estimated mean. The red curve sits neatly over the bars &mdash; that's
  what "the estimate fits" looks like.

**Answer here:**

1. Change `size=200` to `size=20`, run, and watch the red curve vs the bars. Does the fit look
   better or worse with less data? One sentence on the general rule. (Then change it back.)
   &rarr; *your answer*

2. Recall the **Gamma** example from lecture: name one way a Method-of-Moments estimate can
   come out as a value that doesn't even make sense.
   &rarr; *your answer*

---
## Part 3 &middot; Maximum Likelihood (see the 'maximum')

MoM matched averages. **Maximum Likelihood** asks a more direct question:

> Of all the settings I could choose, which makes the data I actually saw the **most probable**?

The **likelihood** is "how probable is my whole dataset under this guess?" &mdash; bigger is better.
One practical tweak: instead of *multiplying* every point's probability (hundreds of tiny
numbers multiply down to zero on a computer), we **add up their logs**. So we maximise the
**log-likelihood**. We'll try many candidate means, score each, and *graph* the scores &mdash; the
peak is the estimate.

In [ ]:
def normal_loglik(x, mu, var):                       # (1) score for one guess (mu, var)
    n = len(x)
    return -0.5*n*np.log(2*np.pi*var) - ((x - mu)**2).sum() / (2*var)

candidates = np.linspace(4.0, 6.0, 401)              # (2) 401 guesses for the mean
scores = np.array([normal_loglik(sample, m, var_hat) for m in candidates])  # (3) score each
best_mu = candidates[scores.argmax()]                # (4) the winner

print(f"best mu by trying lots of values = {best_mu:.3f}")
print(f"sample mean (what MoM gave)      = {sample.mean():.3f}   <- same answer!")

plt.figure(figsize=(7, 4))                           # (5) graph the scores
plt.plot(candidates, scores, lw=2)
plt.axvline(best_mu, color='r', ls='--', label=f'peak at mu = {best_mu:.3f}')
plt.xlabel('candidate mean'); plt.ylabel('log-likelihood (higher = more probable data)')
plt.title('Maximum likelihood = the top of this hill')
plt.legend(); plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** `normal_loglik` takes a guess `(mu, var)` and returns one number: how well that guess
  explains the data. It's just the log of the bell-curve formula, summed over every point.
- **(2)** `np.linspace(4, 6, 401)` makes 401 evenly-spaced candidate means between 4 and 6.
- **(3)** the list comprehension scores *every* candidate &mdash; one log-likelihood per guess.
- **(4)** `scores.argmax()` finds the position of the highest score; `candidates[...]` reads off
  the winning mean.
- **(5)** the plot shows a smooth hill; its peak (dashed line) is the MLE. It lands on the same
  value MoM gave &mdash; two different roads, one destination.

**Answer here:**

1. We *added logs* instead of *multiplying* probabilities. In your own words, what goes wrong if
   you multiply a few hundred tiny probabilities directly on a computer?
   &rarr; *your answer*

2. Here "try lots of values" matched the exact answer. Later some models have **no** exact
   formula and you *must* climb the hill numerically. Name one (a guess is fine).
   &rarr; *your answer*

---
## Part 4 &middot; An estimate is a moving target (see it wobble)

Run the same recipe on a different sample and you get a slightly different answer. To judge an
estimator we ask two plain questions:

- is it **wrong on average**? (that's *bias*)
- does it **home in on the truth as data grows**? (that's *consistency*)

We'll *simulate* the mean-estimator many times at two sample sizes and graph the two clouds of
answers.

In [ ]:
def sampling_dist(n, trials=3000):                   # (1) estimate the mean 'trials' times
    return np.array([rng.normal(true_mu, true_sigma, size=n).mean() for _ in range(trials)])

small = sampling_dist(10)                             # (2) each based on only 10 points
large = sampling_dist(500)                            # (3) each based on 500 points

print(f"n=10 : estimates average {small.mean():.3f}, spread {small.std():.3f}")
print(f"n=500: estimates average {large.mean():.3f}, spread {large.std():.3f}")

plt.figure(figsize=(7, 4))                           # (4) draw both clouds
plt.hist(small, bins=40, density=True, alpha=0.5, label='n = 10')
plt.hist(large, bins=40, density=True, alpha=0.5, label='n = 500')
plt.axvline(true_mu, color='k', ls='--', label=f'truth = {true_mu}')
plt.xlabel('estimated mean'); plt.title('More data -> estimates cluster tighter on the truth')
plt.legend(); plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** `sampling_dist(n)` draws a fresh sample of size `n`, takes its mean, and repeats 3000
  times &mdash; giving 3000 estimates so we can see how they *spread*.
- **(2)&ndash;(3)** we do this for tiny samples (n=10) and big ones (n=500).
- **(4)** the two histograms are the two clouds of answers. Both are centred on the truth (so
  the mean-estimator is **unbiased**), but the n=500 cloud is far narrower &mdash; that narrowing is
  **consistency** in action.

**Answer here:**

1. Both clouds sit on the truth (the mean-estimator is unbiased). Which cloud is *wider*, and
   what does its width tell you about trusting a result from a small sample?
   &rarr; *your answer*

2. In one sentence: what does "consistency" mean, in your own words, after seeing this graph?
   &rarr; *your answer*

---
## Part 5 &middot; The Bias&ndash;Variance Tradeoff (the main event)

Here is the lecture's key identity, in plain words:

> **total error (MSE) = bias&sup2; + variance**

*Bias* is how far off you are **on average**. *Variance* is how much your answer **jumps around**.
You often can't kill both &mdash; pushing one down pushes the other up. The art is minimising the
**sum**.

We'll see this with a concrete choice. To estimate a variance we divide the sum of squared
deviations by *some* number. Divide by $n-1$ and you're **unbiased** but jumpier. Divide by a
*bigger* number and every estimate shrinks &mdash; that adds bias but calms the variance. Let's
sweep the divisor, measure bias&sup2;, variance and MSE by simulation, and graph the tradeoff.

In [ ]:
true_var = true_sigma**2
n = 12
trials = 5000
divisors = np.array([n-1, n, n+1, n+2, n+3, n+4])     # (1) unbiased -> increasingly biased

bias2, variance, mse = [], [], []
for d in divisors:                                    # (2) for each choice of divisor...
    est = np.empty(trials)
    for t in range(trials):                           # (3) ...estimate the variance many times
        s  = rng.normal(true_mu, true_sigma, size=n)
        ss = ((s - s.mean())**2).sum()                # sum of squared deviations
        est[t] = ss / d                               # the estimate for THIS divisor
    bias2.append((est.mean() - true_var)**2)          # (4) how far off on average, squared
    variance.append(est.var())                        # (5) how much the estimates jump around
    mse.append(((est - true_var)**2).mean())          # (6) average squared error = total error
bias2, variance, mse = map(np.array, (bias2, variance, mse))

print(f"{'divisor':>8}{'bias^2':>9}{'variance':>10}{'MSE':>8}{'bias^2+var':>12}")
for i, d in enumerate(divisors):                      # (7) check MSE == bias^2 + variance
    print(f"{d:>8}{bias2[i]:>9.3f}{variance[i]:>10.3f}{mse[i]:>8.3f}{bias2[i]+variance[i]:>12.3f}")
print(f"\nsmallest total error at divisor = {divisors[mse.argmin()]}  (often n+1, not n-1!)")

**Reading the code, line by line:**
- **(1)** we try six divisors, from $n-1$ (the unbiased choice) up to $n+4$ (heavily shrunk).
- **(2)&ndash;(3)** for each divisor we redo the estimate 5000 times on fresh samples of 12 points.
- **(4)** `bias2` = (average estimate &minus; truth)&sup2; &mdash; the *systematic* error.
- **(5)** `variance` = how much those 5000 estimates spread &mdash; the *random* error.
- **(6)** `mse` = the average squared distance from the truth &mdash; the *total* error.
- **(7)** the last two printed columns are equal (up to tiny sampling noise): that **is** the
  identity MSE = bias&sup2; + variance, confirmed by simulation rather than algebra.

In [ ]:
plt.figure(figsize=(7.5, 4.5))
plt.plot(divisors, bias2,    'o-', label='bias$^2$ (rises as we shrink)')
plt.plot(divisors, variance, 's-', label='variance (falls as we shrink)')
plt.plot(divisors, mse,      '^-', lw=2.5, color='crimson', label='MSE = bias$^2$ + variance')
plt.axvline(divisors[mse.argmin()], ls='--', color='gray', alpha=0.7,
            label=f'MSE minimum at divisor {divisors[mse.argmin()]}')
plt.xlabel('divisor used to estimate the variance')
plt.ylabel('error')
plt.title('Bias-variance tradeoff: a little bias can lower the TOTAL error')
plt.legend(); plt.tight_layout(); plt.show()

**What the graph shows:** bias&sup2; climbs and variance drops as the divisor grows &mdash; they pull in
opposite directions. Their sum, the red MSE curve, is **U-shaped**: the best estimator is not
the unbiased one at $n-1$, but a slightly *biased* one further right. That is the whole point of
the tradeoff, and it's exactly why regularisation (week 4) deliberately adds a little bias.

**Answer here:**

1. Read the red MSE curve: is its lowest point at the unbiased divisor $n-1$, or somewhere to
   its right? What does that say about "unbiased is always best"?
   &rarr; *your answer*

2. In your own words, one sentence each: what is **bias**, and what is **variance**?
   &rarr; *your answer*

3. Week 4 (Regularisation) adds a penalty that *shrinks* the model's weights &mdash; on purpose.
   Using this graph, guess in one sentence why deliberately adding bias can help.
   &rarr; *your answer*

---
## Part 6 &middot; When both methods agree exactly

For a **Poisson** &mdash; counts of things, like dengue cases per barangay per week &mdash; MoM and MLE
collapse to the *same* rule: the estimated rate is just the average count. All the machinery
reduces to "take the mean" when the distribution is simple enough.

In [ ]:
true_lambda = 3.5
counts = rng.poisson(true_lambda, size=300)   # 300 count observations
estimate = counts.mean()                      # BOTH MoM and MLE give exactly this

print(f"true rate = {true_lambda}")
print(f"estimate  = {estimate:.3f}   (identical for MoM and MLE)")

**Answer here:**

1. For the Poisson, MoM and MLE are identical. From lecture, name one distribution where they
   are **not**, and which you'd trust more there.
   &rarr; *your answer*

2. Give one real Philippine dataset that is genuinely count-like (Poisson-ish), and one that
   *looks* count-like but would mislead a Poisson model.
   &rarr; *your answer*

---
## Part 7 &middot; Where you actually are

Same as last week &mdash; set the pace honestly. Replace each `-` with:
**solid** / **rusty** / **never really got it**.

| | You |
|---|---|
| Mean and variance in NumPy | - |
| What a "moment" is | - |
| What a likelihood means, in words | - |
| Reading a histogram / a line graph | - |
| The idea MSE = bias&sup2; + variance | - |

**Which part took longest, and where did you get stuck?**
&rarr; *your answer*

**In one plain sentence: why can a slightly biased estimate beat an unbiased one?**
&rarr; *your answer*

---
## Stretch &mdash; optional

Required part is done; nothing below is graded.

### Stretch &middot; Your turn: the Exponential

Waiting times are often **Exponential** &mdash; think minutes between jeepneys at a stop. The MLE
rule is short: **estimated rate = 1 / (average wait)**. Data is below. Write the one line that
estimates the rate, then print the truth next to your estimate.

In [ ]:
true_rate = 0.5
waits = rng.exponential(1 / true_rate, size=250)   # numpy uses the mean, which is 1/rate

# your code here: estimate the rate as 1 / (average wait), then print truth vs estimate


---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to download.

You need a **submit token**: open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token),
sign in, press the button, then paste it when the cell asks. The cell hides what you type, so
the token never gets saved inside your notebook.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "cmsc173", 2

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/cmsc173/lab/2/submit"
    )

nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]
token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 2 submission page](https://portal.latarak.com/course/cmsc173/lab/2/submit) and upload it.

Blank cells are fine and guesses are fine. What is not useful is polishing this until it hides
what you actually knew.